# Improved Training for Diabetic Retinopathy Classification

Notebook ini dibuat untuk mengungguli baseline model (**75.7% top-1 accuracy**) yang ditraining dengan GPU RTX 5070 Ti.

### Analisis Masalah Percobaan Sebelumnya:
- Percobaan 1: CPU, 15 epochs, AdamW `lr0=0.01` → **68.8%** ❌ (lr terlalu besar untuk AdamW)

### Strategi Perbaikan (v2):
1. **Learning Rate Fix**: `lr0=0.001` — nilai yang benar untuk AdamW (10x lebih kecil dari SGD default).
2. **Epochs**: Ditingkatkan ke **50** dengan `patience=15` early stopping.
3. **Image Size**: Dinaikkan ke **320** untuk menangkap detail retina lebih baik.
4. **Augmentasi yang Tepat untuk Medis**:
   - `flipud=0.0` — citra retina tidak di-flip vertikal (tidak valid secara medis)
   - `fliplr=0.5` — flip horizontal tetap valid
   - `degrees=10` — rotasi kecil yang wajar
   - `mixup=0.1` — regularisasi ringan
   - `label_smoothing=0.1` — mencegah over-confident predictions
5. **Optimizer**: AdamW dengan `lr0=0.001` yang benar
6. **Cosine LR Schedule**: Konvergensi yang halus

In [7]:
import os
from pathlib import Path
from ultralytics import YOLO

# Cek apakah dataset sudah tersedia
dataset_path = Path("DDR-dataset-1")

if not dataset_path.exists():
    print("Dataset tidak ditemukan! Pastikan folder 'DDR-dataset-1' ada di direktori yang sama.")
else:
    print(f"Dataset siap di: {dataset_path.absolute()}")

Dataset siap di: f:\telu\semester 6\mulmed\diabetic-retinopathy-classification-for-study-group\DDR-dataset-1


In [8]:
# Load pretrained YOLO11 Nano Classification model
model = YOLO('yolo11n-cls.pt')

# Jalankan Training dengan hyperparameter yang dikoreksi
results = model.train(
    data=str(dataset_path.absolute()),
    epochs=50,              # Lebih banyak dari baseline (5), early stopping akan berhenti jika optimal
    patience=15,            # Stop jika tidak ada improvement setelah 15 epoch
    imgsz=320,              # Lebih besar dari baseline (224)
    batch=16,               # Optimal untuk CPU training
    name="ddr-improved-v2",
    device='cpu',           # Menggunakan CPU
    amp=False,              # Matikan AMP untuk CPU stability
    workers=0,              # Aman untuk Windows
    # === Optimizer ===
    optimizer='AdamW',      # Optimizer modern yang lebih stabil
    lr0=0.001,              # Learning rate yang tepat untuk AdamW (bukan 0.01)
    lrf=0.01,               # Final LR ratio
    cos_lr=True,            # Cosine annealing untuk konvergensi halus
    # === Regularisasi ===
    dropout=0.3,            # Dropout untuk mencegah overfitting
    label_smoothing=0.1,    # Mencegah model terlalu overconfident
    # === Augmentasi yang tepat untuk retina ===
    fliplr=0.5,             # Horizontal flip
    flipud=0.0,             # TIDAK di-flip vertikal (tidak valid medis)
    degrees=10,             # Rotasi ringan
    scale=0.85,             # Scale konservatif
    hsv_h=0.01,             # Variasi hue minimal
    hsv_s=0.5,              # Variasi saturation
    hsv_v=0.3,              # Variasi brightness
    mixup=0.1,              # MixUp regularization ringan
    translate=0.05,         # Translasi minimal
)

WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.46  Python-3.11.9 torch-2.11.0+cpu CPU (Intel Core(TM) i5-10400 2.90GHz)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=f:\telu\semester 6\mulmed\diabetic-retinopathy-classification-for-study-group\DDR-dataset-1, degrees=10, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11n-cls.pt, momentu

In [9]:
top1 = results.results_dict.get('metrics/accuracy_top1', 0)
baseline = 0.757

print("\n" + "="*50)
print("HASIL TRAINING")
print("="*50)
print(f"Top-1 Accuracy (ours)    : {top1:.4f} ({top1*100:.2f}%)")
print(f"Top-1 Accuracy (baseline): {baseline:.4f} ({baseline*100:.2f}%)")
print(f"Selisih                  : {(top1-baseline)*100:+.2f}%")
if top1 > baseline:
    print("BERHASIL MENGUNGGULI BASELINE!")
else:
    print("Belum mengungguli baseline")
print("="*50)
print(f"Hasil disimpan di: {results.save_dir}")


HASIL TRAINING
Top-1 Accuracy (ours)    : 0.7916 (79.16%)
Top-1 Accuracy (baseline): 0.7570 (75.70%)
Selisih                  : +3.46%
BERHASIL MENGUNGGULI BASELINE!
Hasil disimpan di: F:\telu\semester 6\mulmed\diabetic-retinopathy-classification-for-study-group\runs\classify\ddr-improved-v2-2
